# Intake: Perera

The Perera *et al.* (Science 2018) flow Suzuki screen -- 3,072 reactions,
8 phosphine ligands over 4 aryl halides x 3 boron sources x 7 reagents
x 4 solvents.

Source:
[Perera2018_Data_Original.csv](https://raw.githubusercontent.com/Paulilein/Perera2018/main/Perera2018_Data_Original.csv)

This notebook turns that screen into a bundle the engine can train on. Edit the
marked cells; the rest is shared with every other intake notebook.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()            # datasets/perera/
ROOT = HERE.parent.parent              # gp_collab_hub/
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(HERE))

from gpc import prep
from gpc.data import load_bundle

pd.set_option("display.width", 220, "display.max_columns", 80)

OUT = HERE / "inputs"                  # the bundle this notebook writes

# ---- Editable -----------------------------------------------------------
DATASET      = "perera"
DISPLAY_NAME = "Perera"
TARGET       = "Product_Yield_PCT_Area_UV"
GROUP        = "ligand"
CATEGORICAL  = ["Reactant_1_Short_Hand", "Reagent_1_Short_Hand",
                "Solvent_1_Short_Hand", "substrate_pair"]
SOURCE_URL   = ("https://raw.githubusercontent.com/Paulilein/Perera2018/main/"
                "Perera2018_Data_Original.csv")

# "adopt" reuses the verified reactions table already in this folder's bundle.
# "raw"   rebuilds it from the original download -- see the cell below.
SOURCE = "adopt"
RAW = HERE / "raw"
# --------------------------------------------------------------------------
print(f"source mode: {SOURCE}")

## 1. Build the reactions table

The screen's own decisions live here, and they are the part worth editing. As
recorded for this dataset:

* two solvent levels dropped (`MeOH/H2O_V2 9:1`, `THF_V2`) -- reruns under a
  changed protocol;
* four ligand levels dropped (`None`, `dppf`, `dtbpf`, `Xantphos`) -- the bare
  control and three bidentates, which have no Kraken monodentate descriptors;
* `substrate_pair` added as `reactant1_code + "_" + reactant2_code`, so the
  4 x 3 substrate grid is one categorical field rather than two;
* the CSV is `;`-separated with `,` as the decimal mark.

What remains is 8 ligands x 384 reactions = 3,072.

**On `SOURCE = "raw"`:** the reshaping from the original download is not
implemented in this repository -- the code that first produced this table lives
outside it. The cell below raises rather than guessing, and the decisions above
are what it needs to reproduce. `"adopt"` is the working path and gives the
identical, already-verified table.

In [ ]:
if SOURCE == "adopt":
    # The verified table: 3,072 rows with row_id, ligand, kraken_id, the four
    # condition fields and the target already resolved.
    source_table = OUT / "reactions.csv"
    if not source_table.exists():
        raise FileNotFoundError(
            f"{source_table} not found. This is the seed table for the adopt "
            f"path; use SOURCE='raw' once the rebuild below is implemented.")
    reactions = pd.read_csv(source_table, keep_default_na=False)
    reactions[TARGET] = pd.to_numeric(reactions[TARGET])

elif SOURCE == "raw":
    raise NotImplementedError(
        "The Perera raw rebuild is not in this repository. To implement it "
        "here: read the ;-separated CSV with decimal=',', drop the two solvent "
        "and four ligand levels listed above, add substrate_pair, then hand the "
        "frame to prep.add_row_ids and prep.attach_kraken below.")
else:
    raise ValueError(f"SOURCE must be 'adopt' or 'raw', not {SOURCE!r}")

print(f"{len(reactions)} reactions x {reactions.shape[1]} columns")
display(reactions.head(3))

## 2. The ligand mapping

Kraken spells several of these differently from the paper, so the mapping is a
judgement call rather than a string match -- `prep.find_kraken_id("CataCXium")`
shows the candidates for any name you are unsure of.

In [ ]:
# ---- Editable -----------------------------------------------------------
MAPPING = {
    "XPhos": 1, "SPhos": 3, "P(tBu)3": 8, "P(o-Tol)3": 9,
    "CataCXium A": 10, "P(Cy)3": 11, "P(Ph)3": 17, "AmPhos": 216,
}
# --------------------------------------------------------------------------

reactions = prep.attach_kraken(reactions.drop(columns=["kraken_id"], errors="ignore"),
                               GROUP, MAPPING)
if "row_id" not in reactions:
    reactions = prep.add_row_ids(reactions)

identifiers = prep.kraken_identifiers().set_index("id")
display(pd.DataFrame([{"ligand": name, "kraken_id": kid,
                       "kraken_name": identifiers["ligand"].get(kid, "?")}
                      for name, kid in MAPPING.items()]))

## 4. Describe it, then validate

The table below is the record of what this bundle claims about itself: rows and
target statistics per group. Read it before writing -- an unbalanced design
shows up here, and it changes how the LOLO folds should be read (their sizes
follow these counts).

In [ ]:
display(prep.describe(reactions, {"data": {"target": TARGET, "group": GROUP}}))

In [ ]:
# ---- Editable: what this dataset IS -------------------------------------
cfg = prep.bundle_config(
    dataset=DATASET,
    display_name=DISPLAY_NAME,
    target=TARGET,
    group=GROUP,
    categorical=CATEGORICAL,
    reactions=reactions,
    source=SOURCE_URL,
    # models/methods default sensibly: all five sections, and the method list
    # whose matched control has this screen's own group count. Pass explicit
    # lists here to override.
)
# --------------------------------------------------------------------------

display(pd.json_normalize(cfg["data"]).T.rename(columns={0: "value"}))
print("methods:", cfg["evaluation"]["methods"])

# Every check load_bundle will make, run here so a bad bundle fails at the
# point it was built rather than on the cluster three hours into a job.
display(prep.check_bundle(reactions, cfg, prep.kraken_features()))

## 5. Write the bundle

`write_bundle` writes `reactions.csv`, the Kraken descriptor rows for the
ligands this screen uses, `ligand_mapping.csv`, the copied reference PCA and
`config.json`. The PCA is **copied, never refitted** -- PC1..PC4 have to mean
the same axes in every dataset or `pc_top` and `pc_scores` stop being
comparable, which is the point of running them.

In [ ]:
bundle = prep.write_bundle(OUT, reactions, cfg, MAPPING)

# Read it straight back through the engine's own loader: if this succeeds, the
# bundle is usable by `gpc train` exactly as written.
back, ligands, reference, prepared = load_bundle(bundle)
print(f"\nreloaded: {len(back)} rows, {back[cfg['data']['group']].nunique()} "
      f"{cfg['data']['group']}s, {len(ligands)} descriptor rows, "
      f"PCA over {len(reference.columns)} descriptors")
display(pd.DataFrame([{"file": p.name, "KB": round(p.stat().st_size / 1024, 1)}
                      for p in sorted(bundle.iterdir())]))
print("\nNext: build_run.ipynb in the hub root, and pick this dataset.")